# Proyecto final · Predicción de partidos ATP
## Paso 1 · ¿Hace falta ML? El problema y la regla a mano

**Pregunta:** antes de que empiece un partido de nivel ATP Tour, ¿podemos predecir quién lo gana?

- **Una fila** = un partido de cuadro principal del ATP Tour (incluye Grand Slams, Masters, ATP 250/500, Finals y Copa Davis).
- **Etiqueta** = `gana_A`: 1 si el jugador A ganó el partido, 0 si ganó B. Quién es A y quién es B se sortea al azar.
- **Por qué vale la pena predecirla:** es la pregunta central de entrenadores, analistas y medios deportivos. Además tiene un punto de comparación natural y exigente: el ranking ATP, que ya resume el rendimiento del último año.

**Regla a mano (sin ML):** *"gana el jugador con mejor ranking"*. Si un modelo no supera esta regla, no justifica su complejidad.

**Datos:** Jeff Sackmann / Tennis Abstract (tennisabstract.com). Licencia CC BY-NC-SA 4.0: atribución obligatoria y solo uso no comercial.

- El repositorio original `JeffSackmann/tennis_atp` ya no está disponible en GitHub (devuelve 404 desde septiembre de 2026).
- La página de Tennis Abstract no ofrece descarga: su `robots.txt` prohíbe acceder a los archivos de partidos (`/jsmatches/`), así que no la raspamos.
- Usamos el archivo espejo **`Aneeshers/tennis-sackmann-archive`** (Hugging Face y GitHub). Son los mismos CSV, con la misma licencia, tomados de la versión de junio de 2026.

In [1]:
import os
import numpy as np
import pandas as pd

SEMILLA = 42
ANIO_INICIO, ANIO_FIN = 2000, 2026
# Espejo del dataset de Sackmann (el repositorio original fue retirado de GitHub)
URLS = [
    "https://huggingface.co/datasets/Aneeshers/tennis-sackmann-archive/resolve/main/atp/atp_matches_{}.csv",
    "https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/main/atp/atp_matches_{}.csv",
]
CARPETA_LOCAL = os.environ.get("TENIS_DATOS", "datos")  # si ya descargaron los CSV, se leen de aquí

### Carga de los datos
Se lee un archivo por temporada (`atp_matches_YYYY.csv`). Si existe la carpeta `datos/` con los CSV, se usan esos. Si no, se descargan del espejo y se guardan en `datos/`, de modo que la segunda vez (y el día de la exposición) no hace falta internet.

In [2]:
os.makedirs(CARPETA_LOCAL, exist_ok=True)

def leer_anio(anio):
    local = os.path.join(CARPETA_LOCAL, f"atp_matches_{anio}.csv")
    if os.path.exists(local):
        return pd.read_csv(local, low_memory=False)
    for url in URLS:
        try:
            t = pd.read_csv(url.format(anio), low_memory=False)
            t.to_csv(local, index=False)
            return t
        except Exception:
            continue
    print(f"  {anio}: no disponible")
    return None

tablas = [t for a in range(ANIO_INICIO, ANIO_FIN + 1) if (t := leer_anio(a)) is not None]
partidos = pd.concat(tablas, ignore_index=True)
partidos["tourney_date"] = pd.to_datetime(partidos["tourney_date"].astype(str), format="%Y%m%d")
partidos["anio"] = partidos["tourney_date"].dt.year

print(f"Filas (partidos): {len(partidos):,}")
print(f"Columnas: {partidos.shape[1]}")
print(f"Periodo: {partidos['tourney_date'].min().date()} a {partidos['tourney_date'].max().date()}")

Filas (partidos): 79,299
Columnas: 50
Periodo: 2000-01-03 a 2026-05-25


In [3]:
# Así se ven los datos crudos: todo viene como ganador (winner_*) / perdedor (loser_*)
partidos[["tourney_name", "tourney_date", "surface", "round",
          "winner_name", "winner_rank", "loser_name", "loser_rank", "score"]].head()

,tourney_name,tourney_date,surface,round,winner_name,winner_rank,loser_name,loser_rank,score
0,Auckland,2000-01-10,Hard,R32,Tommy Haas,11.0,Jeff Tarango,63.0,7-5 4-6 7-5
1,Auckland,2000-01-10,Hard,R32,Juan Balcells,211.0,Franco Squillari,49.0,7-5 7-5
2,Auckland,2000-01-10,Hard,R32,Alberto Martin,48.0,Alberto Berasategui,59.0,6-3 6-1
3,Auckland,2000-01-10,Hard,R32,Juan Carlos Ferrero,45.0,Roger Federer,61.0,6-4 6-4
4,Auckland,2000-01-10,Hard,R32,Michael Sell,167.0,Nicolas Escude,34.0,0-6 7-6(7) 6-1


### El problema de las columnas `winner_` / `loser_`
El ganador siempre está del mismo lado. Si entrenáramos con esa estructura, la etiqueta sería siempre 1 y cualquier modelo "acertaría" el 100 %.

**Solución:** para cada partido se sortea (con semilla fija) si el ganador queda como jugador A o como jugador B. Así la etiqueta queda balanceada, cerca de 50/50.

In [4]:
rng = np.random.default_rng(SEMILLA)
ganador_es_A = rng.random(len(partidos)) < 0.5

df = pd.DataFrame({
    "fecha": partidos["tourney_date"],
    "anio": partidos["anio"],
    "torneo": partidos["tourney_name"],
    "nivel": partidos["tourney_level"],
    "superficie": partidos["surface"],
    "ronda": partidos["round"],
    "score": partidos["score"],
})
for col in ["name", "rank", "rank_points"]:
    df[f"A_{col}"] = np.where(ganador_es_A, partidos[f"winner_{col}"], partidos[f"loser_{col}"])
    df[f"B_{col}"] = np.where(ganador_es_A, partidos[f"loser_{col}"], partidos[f"winner_{col}"])
df["gana_A"] = ganador_es_A.astype(int)

print("Proporción de partidos que gana A:", round(df["gana_A"].mean(), 4))
df.head()

Proporción de partidos que gana A: 0.497


,fecha,anio,torneo,nivel,superficie,ronda,score,A_name,B_name,A_rank,B_rank,A_rank_points,B_rank_points,gana_A
0,2000-01-10,2000,Auckland,A,Hard,R32,7-5 4-6 7-5,Jeff Tarango,Tommy Haas,63.0,11.0,595.0,1612.0,0
1,2000-01-10,2000,Auckland,A,Hard,R32,7-5 7-5,Juan Balcells,Franco Squillari,211.0,49.0,157.0,723.0,1
2,2000-01-10,2000,Auckland,A,Hard,R32,6-3 6-1,Alberto Berasategui,Alberto Martin,59.0,48.0,649.0,726.0,0
3,2000-01-10,2000,Auckland,A,Hard,R32,6-4 6-4,Roger Federer,Juan Carlos Ferrero,61.0,45.0,616.0,768.0,0
4,2000-01-10,2000,Auckland,A,Hard,R32,0-6 7-6(7) 6-1,Michael Sell,Nicolas Escude,167.0,34.0,219.0,873.0,1


### Línea base y regla a mano
- **Línea base:** predecir siempre "gana A". Con la etiqueta balanceada, debe dar cerca de 50 %.
- **Regla 1 · ranking:** gana el de mejor ranking (número más bajo).
- **Regla 2 · puntos ATP:** gana el de más puntos.

Si a un jugador le falta el ranking (por ejemplo, porque no está clasificado), se asume que el clasificado es mejor. Si a ambos les falta, la regla no puede decidir y se cuenta como predicción "gana A".

In [5]:
def regla_ranking(d):
    rA = d["A_rank"].fillna(np.inf)
    rB = d["B_rank"].fillna(np.inf)
    return (rA < rB).astype(int).where(rA != rB, 1)

def regla_puntos(d):
    pA = d["A_rank_points"].fillna(0)
    pB = d["B_rank_points"].fillna(0)
    return (pA > pB).astype(int).where(pA != pB, 1)

df["pred_base"] = 1
df["pred_ranking"] = regla_ranking(df)
df["pred_puntos"] = regla_puntos(df)

ambos_con_ranking = df["A_rank"].notna() & df["B_rank"].notna()
print(f"Partidos con ranking de ambos jugadores: {ambos_con_ranking.mean():.1%}")

resumen = pd.DataFrame({
    "Exactitud (todos los partidos)": [
        (df[p] == df["gana_A"]).mean() for p in ["pred_base", "pred_ranking", "pred_puntos"]],
    "Exactitud (ambos con ranking)": [
        (df.loc[ambos_con_ranking, p] == df.loc[ambos_con_ranking, "gana_A"]).mean()
        for p in ["pred_base", "pred_ranking", "pred_puntos"]],
}, index=["Línea base (siempre A)", "Regla: mejor ranking", "Regla: más puntos ATP"])
(resumen * 100).round(1)

Partidos con ranking de ambos jugadores: 97.6%


,Exactitud (todos los partidos),Exactitud (ambos con ranking)
Línea base (siempre A),49.7,49.7
Regla: mejor ranking,65.6,65.4
Regla: más puntos ATP,65.6,65.4


### ¿La regla funciona igual en todos los casos?
Si la regla acierta mucho más en unos contextos que en otros, hay información que un modelo puede aprovechar. Eso es lo que justifica usar ML.

In [6]:
acierto = (df["pred_ranking"] == df["gana_A"])

print("Por superficie:")
display((acierto.groupby(df["superficie"]).agg(["mean", "size"])
         .rename(columns={"mean": "exactitud", "size": "partidos"})
         .assign(exactitud=lambda t: (t["exactitud"] * 100).round(1))))

print("Por nivel del torneo (G = Grand Slam, M = Masters, A = ATP 250/500, F = Finals, D = Copa Davis):")
display((acierto.groupby(df["nivel"]).agg(["mean", "size"])
         .rename(columns={"mean": "exactitud", "size": "partidos"})
         .assign(exactitud=lambda t: (t["exactitud"] * 100).round(1))))

Por superficie:


,exactitud,partidos
superficie,,
Carpet,64.2,2039
Clay,65.0,25746
Grass,65.0,8000
Hard,66.2,43461


Por nivel del torneo (G = Grand Slam, M = Masters, A = ATP 250/500, F = Finals, D = Copa Davis):


,exactitud,partidos
nivel,,
A,63.3,42858
D,72.2,7367
F,64.7,467
G,71.2,13335
M,64.2,15208
O,75.0,64


In [7]:
# ¿Qué tan bien funciona la regla según la diferencia de ranking?
dif = (df["A_rank"] - df["B_rank"]).abs()
tramos = pd.cut(dif, bins=[0, 10, 25, 50, 100, 250, np.inf],
                labels=["1-10", "11-25", "26-50", "51-100", "101-250", ">250"])
(acierto[ambos_con_ranking].groupby(tramos[ambos_con_ranking], observed=True)
 .agg(exactitud="mean", partidos="size")
 .assign(exactitud=lambda t: (t["exactitud"] * 100).round(1)))

,exactitud,partidos
1-10,55.0,10729
11-25,59.5,14350
26-50,64.0,18404
51-100,69.4,17787
101-250,72.4,10876
>250,79.6,5281


In [8]:
# Exactitud de la regla por año: ¿es estable en el tiempo?
(acierto.groupby(df["anio"]).mean() * 100).round(1).to_frame("exactitud regla ranking (%)").T

anio,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
exactitud regla ranking (%),64.2,63.4,64.0,65.1,64.8,66.8,65.8,65.9,67.8,68.2,...,65.5,64.2,61.8,64.4,63.9,64.8,63.3,63.9,64.8,63.9


### Conclusión del paso 1
Datos: unos 79.300 partidos del ATP Tour entre 2000 y junio de 2026. El 97,6 % tiene el ranking de ambos jugadores.
 
| Método | Exactitud |
|---|---|
| Línea base (siempre gana A) | 49,7 % |
| Regla: mejor ranking | 65,6 % |
| Regla: más puntos ATP | 65,6 % |
 
- **La etiqueta está balanceada** (49,7 %), así que el sorteo A/B funcionó.
- **La regla del ranking acierta el 65,6 %** y es estable año a año (entre 62 % y 68 %). En los años que usaremos como prueba (2023–2026) ronda el 64 %: esa es la cifra que el modelo debe superar.
- **Ranking y puntos dan exactamente lo mismo**, porque el ranking es el orden de los puntos. Son redundantes como regla. En el modelo, los puntos aportan algo que el ranking no: la *magnitud* de la diferencia.
- **¿Por qué ML?** La regla acierta el 79,6 % cuando la diferencia de ranking supera 250 puestos, pero solo el 55,0 % entre jugadores separados por 1 a 10 puestos, casi una moneda al aire. En esos partidos parejos hace falta más información: Elo por superficie, forma reciente, historial directo y cansancio.
- **Hallazgo:** la regla acierta el 71,2 % en Grand Slams (a 5 sets), frente al 63,3 % en ATP 250/500. Los partidos más largos favorecen al mejor jugador, así que `best_of` entra como variable.
 
**Pendientes para la limpieza:** superficie "Carpet" (2.039 partidos, en desuso), unos 53 partidos sin superficie, 64 partidos de nivel "O" y los retiros/walkovers en `score`.
 
**Siguiente paso:** unir las tablas de jugadores y rankings, y limpiar.